# 09｜SIRIUS Development 正负模式输入生成

本步骤只使用冻结的 **1,230 个、20 ppm Development 化合物**及其公共谱图，生成正、负模式分开的 SIRIUS `.ms` 输入。ChemVista 谱图保持锁定；输入不包含真实分子结构或正确分子式。

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Python:', sys.executable)
print('Project:', ROOT)

In [ ]:
from pbt_msms.sirius_export import run_export, OUTPUT_DIR

manifest = run_export()
manifest['counts']

In [ ]:
qc = pd.read_csv(OUTPUT_DIR / 'development_sirius_input_qc_checks.csv')
display(qc)
assert not qc['status'].eq('FAIL').any()
print('全部导出 QC：PASS')

In [ ]:
features = pd.read_csv(OUTPUT_DIR / 'development_sirius_feature_manifest.csv.gz')
summary = features.groupby('ion_mode').agg(
    feature_n=('sirius_feature_id', 'nunique'),
    compound_n=('inchikey_2d', 'nunique'),
    selected_spectrum_n=('selected_spectrum_n', 'sum'),
).reset_index()
display(summary)
print('运行说明:', OUTPUT_DIR / 'SIRIUS_运行说明.md')

## 下一步

在 SIRIUS 6 GUI 中分别导入 `positive_ms` 和 `negative_ms` 目录。先核对导入 feature 数与上表完全一致，再计算 molecular formula 和 CSI:FingerID fingerprint。主分析不得填写正确分子式。